In [ ]:
from docx import Document
import re
import os
from tqdm.auto import tqdm
from collections import defaultdict
import spacy
import textwrap
import pandas as pd
import plotly.express as px

from SARVI.config import paths

# !python -m spacy download es_core_news_lg
nlp = spacy.load("es_core_news_lg")

In [3]:
def plot_errores_apilados_por_archivo(
    total: pd.Series,
    errores: pd.Series,
    title: str = "Correctos vs Errores por archivo (apilado)",
    y_max: float | None = None,
    top: int | None = None,
    ordenar_por: str = "porcentaje",   # "porcentaje" | "total" | "errores"
    asc: bool = False,
    rot_x: int = 60
):
    # Alinear y sanear
    errores_aligned = errores.reindex(total.index, fill_value=0).clip(upper=total)
    correctos = (total - errores_aligned).clip(lower=0)

    # Base
    base = pd.DataFrame({
        "nombre_archivo": total.index.astype(str),
        "total": total.values,
        "errores": errores_aligned.values,
        "correctos": correctos.values,
    })
    base["porcentaje"] = (base["errores"] / base["total"]).where(base["total"] > 0, 0) * 100

    # Orden / top
    if ordenar_por not in {"porcentaje", "total", "errores"}:
        ordenar_por = "porcentaje"
    base = base.sort_values(ordenar_por, ascending=asc)
    if top is not None:
        base = base.head(top)

    # Largo para px.bar (True/False como en tu hist original)
    df_long = base.melt(
        id_vars=["nombre_archivo", "total", "porcentaje"],
        value_vars=["correctos", "errores"],
        var_name="tipo",
        value_name="cantidad"
    )
    # True -> Correctos (verde), False -> Errores (rojo)
    df_long["binario"] = df_long["tipo"].map({"correctos": True, "errores": False})

    fig = px.bar(
        df_long,
        x="nombre_archivo",
        y="cantidad",
        color="binario",
        color_discrete_map={True: "green", False: "red"},
        labels={
            "nombre_archivo": "nombre_archivo",
            "cantidad": "Frecuencia",
            "binario": "Tipo (True=Correctos, False=Errores)"
        },
        title=title
    )

    # Apilado (amontonado)
    fig.update_layout(barmode="relative", bargap=0)

    # Anotar % de error encima del total (cima de la pila)
    for _, r in base.iterrows():
        fig.add_annotation(
            x=r["nombre_archivo"],
            y=r["total"],
            text=f'{r["porcentaje"]:.1f}%',
            showarrow=False,
            yshift=8
        )

    # Ejes y leyenda
    if y_max is not None:
        fig.update_yaxes(range=[0, y_max], autorange=False, title="Frecuencia")
    else:
        fig.update_yaxes(autorange=True, title="Frecuencia")
    fig.update_xaxes(tickangle=rot_x)
    fig.update_layout(
        height=560,
        margin=dict(t=90),
        legend_title_text="Tipo"
    )

    # Totales globales en la leyenda (opcional)
    sum_corr = int(base["correctos"].sum())
    sum_err  = int(base["errores"].sum())
    def _rename_trace(t):
        if str(t.name).strip().lower() == "true":
            t.name = f"True (Correctos={sum_corr})"
        else:
            t.name = f"False (Errores={sum_err})"
    fig.for_each_trace(_rename_trace)

    fig.update_xaxes(categoryorder="array",
                 categoryarray=sorted(base["nombre_archivo"].astype(str)))


    fig.show()


# Leer DOCX

In [4]:
def parse_docx_outline(path: str, max_heading_len: int = 80, bold_ratio_threshold: float = 0.8):
    """
    - Heading reales (Heading/Título 1..9) definen la jerarquía.
    - Pseudo-títulos: párrafos cortos en negrita (no lista) o fragmento inicial en negrita seguido de contenido en el mismo párrafo (p. ej. 'Subsección: contenido...').
        Siempre cuelgan como HERMANOS bajo el último Heading real.
    - Se evita detectar listas (viñetas/numeradas).
    """
    doc = Document(path)
    root = {"title": "ROOT", "level": 0, "content": "", "children": []}
    stack = [root]
    last_true_heading_index = 0  # índice en stack del último heading real

    LIST_PREFIX_RE = re.compile(r"""
        ^(
            [-–•●·]            # viñetas comunes
            |
            \d+[\.\)]          # 1. o 1)
            |
            [a-zA-Z][\.\)]     # a. o a)
        )\s+
    """, re.VERBOSE)

    def heading_level(style_name: str):
        name = (style_name or "").lower()
        for i in range(1, 10):
            if f"heading {i}" in name or f"título {i}" in name or f"titulo {i}" in name:
                return i
        return None

    def run_is_bold(run):
        if run.bold is True:
            return True
        try:
            if getattr(getattr(run, "style", None), "font", None):
                if run.style.font.bold is True:
                    return True
        except Exception:
            pass
        return False

    def paragraph_bold_ratio(p):
        total = bold = 0
        for r in p.runs:
            t = r.text or ""
            n = len(t)
            total += n
            if n and run_is_bold(r):
                bold += n
        if total == 0:
            try:
                if p.style and p.style.font and p.style.font.bold:
                    return 1.0
            except Exception:
                pass
            return 0.0
        return bold / total

    def is_list_paragraph(p):
        try:
            pPr = p._p.pPr
            if pPr is not None and pPr.numPr is not None:
                return True
        except Exception:
            pass
        sty = (getattr(p, "style", None).name or "").lower() if getattr(p, "style", None) else ""
        if "list" in sty or "lista" in sty:
            return True
        txt = (p.text or "").strip()
        if LIST_PREFIX_RE.match(txt):
            return True
        return False

    def split_leading_bold_heading(p):
        """
        Si el párrafo empieza con un bloque en negrita (contiguo) que actúa como título,
        devuelve (heading_text, trailing_content). Soporta que ':' esté en negrita o no.
        """
        parts = [(r.text or "", run_is_bold(r)) for r in p.runs if (r.text or "")]
        if not parts:
            return None, None

        # Reconstruye texto y máscara de negrita por carácter
        s = ""
        mask = []
        for t, b in parts:
            s += t
            mask.extend([b] * len(t))

        # Saltar espacios iniciales
        i = 0
        while i < len(s) and s[i].isspace():
            i += 1
        start = i

        # Tomar tramo inicial en negrita contiguo
        while i < len(s) and mask[i] and s[i] != "\n":
            i += 1

        leading = s[start:i].strip()
        if not leading:
            return None, None

        # ¿dos casos válidos? (a) el bloque bold acaba con ':' o
        # (b) el siguiente no-blanco inmediato es ':' aunque no sea bold
        j = i
        while j < len(s) and s[j].isspace():
            j += 1
        next_is_colon = (j < len(s) and s[j] == ":")

        ends_with_colon = leading.endswith(":")
        heading_core = leading.rstrip(":").strip()

        # Heurística: título si (tiene ':' al final o justo después) o si es corto
        if ends_with_colon or next_is_colon or len(heading_core) <= max_heading_len:
            # Consumir el ':' si estaba justo después
            end_idx = j + 1 if next_is_colon else i
            trailing = s[end_idx:].lstrip()
            # Evitar falsos positivos largos
            if len(heading_core) <= max_heading_len and not is_list_paragraph(p):
                # Si el párrafo entero es bold y no tiene contenido adicional,
                # dejamos que lo trate 'is_pseudo_heading'; aquí nos centramos en "bold seguido de contenido"
                if trailing:
                    return heading_core, trailing
        return None, None

    def is_pseudo_heading(p):
        # Para párrafos enteros que son “título oficioso”
        text = (p.text or "").strip()
        if not text or is_list_paragraph(p) or len(text) > max_heading_len:
            return False
        if text.endswith(":"):
            return True
        if paragraph_bold_ratio(p) >= bold_ratio_threshold and text.count(".") <= 1:
            return True
        return False

    for p in doc.paragraphs:
        text = (p.text or "").strip()
        if not text:
            continue

        lvl = heading_level(p.style.name if p.style else "")

        if lvl and text:
            # Heading real
            while stack and stack[-1]["level"] >= lvl:
                stack.pop()
            node = {"title": text, "level": lvl, "content": "", "children": []}
            stack[-1]["children"].append(node)
            stack.append(node)
            last_true_heading_index = len(stack) - 1

        else:
            # 1) Caso: "Bold heading: contenido..." en el MISMO párrafo
            h_text, trailing = split_leading_bold_heading(p)
            if h_text:
                parent_idx = last_true_heading_index  # hermano bajo el último heading real
                while len(stack) - 1 > parent_idx:
                    stack.pop()
                parent = stack[parent_idx]
                lvl_new = min(parent["level"] + 1, 9)
                node = {"title": h_text, "level": lvl_new, "content": "", "children": []}
                parent["children"].append(node)
                stack.append(node)
                if trailing:
                    stack[-1]["content"] += trailing + "\n"
                continue

            # 2) Párrafo entero como pseudo-título
            if is_pseudo_heading(p):
                parent_idx = last_true_heading_index
                while len(stack) - 1 > parent_idx:
                    stack.pop()
                parent = stack[parent_idx]
                lvl_new = min(parent["level"] + 1, 9)
                node = {"title": text.rstrip(":"), "level": lvl_new, "content": "", "children": []}
                parent["children"].append(node)
                stack.append(node)
            else:
                # 3) Contenido normal
                stack[-1]["content"] += (p.text + "\n")

    return root["children"]

In [5]:
folder = paths.data_input

estructura = defaultdict(list)

for f in tqdm(os.listdir(folder), desc="Leyendo DOCX", unit="DOCX"):
    estructura[re.split(".docx",f)[0]] = parse_docx_outline(os.path.join(folder, f))

Leyendo DOCX:   0%|          | 0/200 [00:00<?, ?DOCX/s]

# CBOW

In [6]:
def cargar_docx(path: str) -> str:
    """
    Load the `.docx` archive and transform it into a plain `str`

    Parameters
    ----------
        `path`: str
            - Path of the single `.docx` archive

    Returns
    -------
        `texto`: str
            - Plain text of the original doc
    """
    doc = Document(path)
    texto = "\n".join([p.text for p in doc.paragraphs if p.text.strip() != ""])
    texto = textwrap.dedent(texto)
    return texto

In [7]:
folder = paths.data_input
docx = defaultdict(list)

for f in tqdm(os.listdir(folder), desc="Leyendo DOCX", unit="DOCX"):
    txt = cargar_docx(os.path.join(folder, f))
    docx[re.split(".docx",f)[0]] = []
    for token in nlp(txt):
        if not (token.is_stop or token.is_punct or token.is_space):
            docx[re.split(".docx",f)[0]].append(token.text.lower()) 

Leyendo DOCX:   0%|          | 0/200 [00:00<?, ?DOCX/s]

In [8]:
from collections import Counter

bow_full = Counter()
bow_per_doc = defaultdict(Counter)

for f, doc in tqdm(docx.items(), desc="Crando BoW", unit="DOCX"):
    bow_full.update(doc)
    bow_per_doc[f].update(doc)

Crando BoW:   0%|          | 0/200 [00:00<?, ?DOCX/s]

# Comprobar - Errores Diagnosticos

- **any** -> **ALGUNA** palabra **NO** se encuentra -> Fallo
- **all** -> **NO** se encuentra **NINGUNA** palabra -> Fallo

In [27]:
import pandas as pd
import ftfy

data = pd.read_excel(paths.data_output / "df_COMPLETE_EVALUATION_OLLAMA_MEDGEMMA_FINAL.xlsx").map(lambda x: ftfy.fix_text(x) if isinstance(x, str) else x).drop_duplicates()
print(len(data))
data = data.drop_duplicates(subset=["diagnostico_predicted", "CIE10_predicted", "pertenencia", "nombre_archivo"]).reset_index(drop=True)
print(len(data))

1488
1289


In [19]:
error_predicted = defaultdict(str)
error_nearest = defaultdict(str)

for i,(_,row) in tqdm(enumerate(data.iterrows()), desc="Analizando el DataFrame", unit="Fila", total=len(data)):
    if all(bow_per_doc[row["nombre_archivo"]].get(t.text.lower(), 0) == 0 for t in nlp(row["diagnostico_predicted"]) if not (t.is_space or t.is_punct)):
        error_predicted[i] = row["diagnostico_nearest"]
    if any(bow_full.get(t.text.lower(), 0) == 0 for t in nlp(row["diagnostico_nearest"]) if not (t.is_space or t.is_punct)):
        error_nearest[i] = row["diagnostico_nearest"]

print(f"\033[1mDiagnosticos predicted\033[0m:\n\t✅ {len(data)-len(error_predicted)}\n\t❌ {len(error_predicted)}")
print(f"\033[1mPor informe\033[0m:\n\t✅ {len(data["nombre_archivo"].unique().tolist())-len(data.iloc[list(error_predicted.keys())]["nombre_archivo"].unique().tolist())}\n\t❌ {len(data.iloc[list(error_predicted.keys())]["nombre_archivo"].unique().tolist())}")

print(f"\n\033[1mDiagnosticos nearest\033[0m:\n\t✅ {len(data)-len(error_nearest)}\n\t❌ {len(error_nearest)}")
print(f"\033[1mPor informe\033[0m:\n\t✅ {len(data["nombre_archivo"].unique().tolist())-len(data.iloc[list(error_nearest.keys())]["nombre_archivo"].unique().tolist())}\n\t❌ {len(data.iloc[list(error_nearest.keys())]["nombre_archivo"].unique().tolist())}")

Analizando el DataFrame:   0%|          | 0/1289 [00:00<?, ?Fila/s]

Diagnosticos predicted:
	✅ 1231
	❌ 58
Por informe:
	✅ 135
	❌ 23

Diagnosticos nearest:
	✅ 204
	❌ 1085
Por informe:
	✅ 1
	❌ 157


In [20]:
# total = data["nombre_archivo"].value_counts(dropna=False).sort_index()
# errores = data.iloc[list(error_predicted.keys())]["nombre_archivo"].value_counts(dropna=False)

# errores_aligned = errores.reindex(total.index, fill_value=0)
# porcentaje = (errores_aligned / total) * 100

# res = pd.DataFrame({
#     "total": total,
#     "errores": errores_aligned,
#     "porcentaje": porcentaje
# }).sort_values("porcentaje", ascending=False)


In [21]:
total = data["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.str.fullmatch(r"ABDM\d+", na=False)]
errores = data.iloc[list(error_predicted.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="ABDM", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

total = data["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.str.fullmatch(r"ICL\d+", na=False)]
errores = data.iloc[list(error_predicted.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="ICL", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

total = data["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.str.fullmatch(r"MLAA\d+", na=False)]
errores = data.iloc[list(error_predicted.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="MLAA", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

total = data["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.str.fullmatch(r"OBR\d+", na=False)]
errores = data.iloc[list(error_predicted.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="OBR", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

In [22]:
total = data["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.str.fullmatch(r"ABDM\d+", na=False)]
errores = data.iloc[list(error_nearest.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="ABDM", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

total = data["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.str.fullmatch(r"ICL\d+", na=False)]
errores = data.iloc[list(error_nearest.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="ICL", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

total = data["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.str.fullmatch(r"MLAA\d+", na=False)]
errores = data.iloc[list(error_nearest.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="MLAA", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

total = data["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.str.fullmatch(r"OBR\d+", na=False)]
errores = data.iloc[list(error_nearest.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="OBR", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

# Comprobacion - Pertenencia

In [23]:
def recorrer_nodos(nodos, nivel=0):
    for nodo in nodos:
        if "familia" in nodo['title'].lower():
            return nodo["content"]
        if "children" in nodo and isinstance(nodo["children"], list):
            resultado = recorrer_nodos(nodo["children"], nivel + 1)
            if resultado is not None:
                return resultado

In [24]:
aux = defaultdict(int)
pertenencia_posible = defaultdict(Counter)
for name, e in estructura.items():
    result = recorrer_nodos(e)
    if None == result:
        aux[re.split(r"\d", name)[0]] += 1
    else:
        for token in nlp(result):
            total = []
            if not (token.is_stop or token.is_punct or token.is_space):
                total.append(token.text.lower()) 
            pertenencia_posible[name].update(total)
print(aux)
print(len(estructura))

defaultdict(<class 'int'>, {'OBR': 50, 'MLAA': 11, 'ICL': 2})
200


In [25]:
errores_pertenencia = defaultdict(str)
familiar_count = 0

for i,(_,row) in tqdm(enumerate(data.iterrows()), desc="Analizando el DataFrame", unit="Fila", total=len(data)):
    if row["nombre_archivo"] in pertenencia_posible.keys() and row["pertenencia"] == "familiar":
        familiar_count += 1
        if all(pertenencia_posible[row["nombre_archivo"]].get(t.text.lower(), 0) == 0 for t in nlp(row["diagnostico_predicted"]) if not (t.is_space or t.is_punct)):
            errores_pertenencia[i] = row["diagnostico_predicted"]

print(f"\033[1mDiagnosticos predicted (total {familiar_count})\033[0m:\n\t✅ {familiar_count-len(errores_pertenencia)}\n\t❌ {len(errores_pertenencia)}")
print(f"\033[1mPor informe (total {len(pertenencia_posible.keys())})\033[0m:\n\t✅ {len(pertenencia_posible.keys())-len(data.iloc[list(errores_pertenencia.keys())]["nombre_archivo"].unique().tolist())}\n\t❌ {len(data.iloc[list(errores_pertenencia.keys())]["nombre_archivo"].unique().tolist())}")

Analizando el DataFrame:   0%|          | 0/1289 [00:00<?, ?Fila/s]

Diagnosticos predicted (total 114):
	✅ 75
	❌ 39
Por informe (total 137):
	✅ 124
	❌ 13


In [26]:
total = data[data["pertenencia"] == "familiar"]["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.isin(pertenencia_posible.keys())]
total = total[total.index.str.fullmatch(r"ABDM\d+", na=False)]
errores = data.iloc[list(errores_pertenencia.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="ABDM", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

total = data[data["pertenencia"] == "familiar"]["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.isin(pertenencia_posible.keys())]
total = total[total.index.str.fullmatch(r"ICL\d+", na=False)]
errores = data.iloc[list(errores_pertenencia.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="ICL", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

total = data[data["pertenencia"] == "familiar"]["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.isin(pertenencia_posible.keys())]
total = total[total.index.str.fullmatch(r"MLAA\d+", na=False)]
errores = data.iloc[list(errores_pertenencia.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="MLAA", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######

total = data[data["pertenencia"] == "familiar"]["nombre_archivo"].value_counts(dropna=False).sort_index()
total = total[total.index.isin(pertenencia_posible.keys())]
total = total[total.index.str.fullmatch(r"OBR\d+", na=False)]
errores = data.iloc[list(errores_pertenencia.keys())]["nombre_archivo"].value_counts(dropna=False)
errores_aligned = errores.reindex(total.index, fill_value=0)
plot_errores_apilados_por_archivo(total, errores_aligned, title="OBR", y_max=None, top=None, ordenar_por="porcentaje", asc=False)

#######